In [1]:
# -*- coding: utf-8 -*-
"""
PDE:
    u_t + a u_x = nu u_xx,   (x,t) in [0,1] x [0,0.5]

Parameter ranges:
    a  in [0.5, 1.0]
    nu in [0.01, 0.05]

Exact benchmark family used for IC/BC/evaluation:
    u(x,t;a,nu) = 1/sqrt(4t+1) * exp( -(x - 0.2 - a t)^2 / (nu (4t+1)) )

"""

import os
import csv
import math
import random
import argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# ============================================================
# Reproducibility
# ============================================================
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# Config
# ============================================================
@dataclass
class ProblemConfig:
    x_min: float = 0.0
    x_max: float = 1.0
    t_min: float = 0.0
    t_max: float = 0.5
    a_min: float = 0.5
    a_max: float = 1.0
    nu_min: float = 0.01
    nu_max: float = 0.05
    nu_easy: float = 0.03
    x0_center: float = 0.2


CFG = ProblemConfig()


# ============================================================
# Exact solution / IC
# ============================================================
def exact_advecdiff_torch(x: torch.Tensor, t: torch.Tensor, a: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    four_t1 = 4.0 * t + 1.0
    return torch.rsqrt(four_t1) * torch.exp(-((x - CFG.x0_center - a * t) ** 2) / (nu * four_t1))


def exact_advecdiff_np(x: np.ndarray, t: np.ndarray, a: float, nu: float) -> np.ndarray:
    four_t1 = 4.0 * t + 1.0
    return (1.0 / np.sqrt(four_t1)) * np.exp(-((x - CFG.x0_center - a * t) ** 2) / (nu * four_t1))


def initial_condition_torch(x: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    # exact solution at t=0
    return torch.exp(-((x - CFG.x0_center) ** 2) / nu)


# ============================================================
# Fourier features in time
# ============================================================
def time_fourier_features(t: torch.Tensor, num_freqs: int = 4, t_scale: float = 1.0) -> torch.Tensor:
    """
    t: (...,1)
    returns [..., 1 + 2*num_freqs]
    """
    feats = [t]
    for k in range(1, num_freqs + 1):
        omega = 2.0 * math.pi * k / t_scale
        feats.append(torch.sin(omega * t))
        feats.append(torch.cos(omega * t))
    return torch.cat(feats, dim=-1)


# ============================================================
# Model
# ============================================================
class DynamicBasisMetaSPINN_AdvecDiff(nn.Module):
    def __init__(
        self,
        M: int = 48,
        hidden: int = 128,
        num_freqs: int = 4,
        h_min: float = 0.015,
        h_max: float = 0.35,
    ):
        super().__init__()
        self.M = M
        self.num_freqs = num_freqs
        self.h_min = h_min
        self.h_max = h_max

        in_dim = 1 + 2 * num_freqs + 2  # time Fourier + (a, nu)

        self.temporal_net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 3 * M),
        )

        # Learned static spatial anchor distribution.
        # The time/parameter network predicts offsets around these anchors.
        self.anchor_raw = nn.Parameter(torch.linspace(-2.0, 2.0, M))
        self.amp_scale = nn.Parameter(0.1 * torch.ones(M))

    def normalize_params(self, a: torch.Tensor, nu: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        a_mean = 0.5 * (CFG.a_min + CFG.a_max)
        a_std = max(0.5 * (CFG.a_max - CFG.a_min), 1e-8)
        nu_mean = 0.5 * (CFG.nu_min + CFG.nu_max)
        nu_std = max(0.5 * (CFG.nu_max - CFG.nu_min), 1e-8)
        a_n = (a - a_mean) / a_std
        nu_n = (nu - nu_mean) / nu_std
        return a_n, nu_n

    def dynamic_params(self, t: torch.Tensor, a: torch.Tensor, nu: torch.Tensor):
        """
        t, a, nu: (N,1)
        returns alpha, xi, h each of shape (N,M)
        """
        a_n, nu_n = self.normalize_params(a, nu)
        t_feats = time_fourier_features(t, num_freqs=self.num_freqs, t_scale=max(CFG.t_max, 1e-8))
        inp = torch.cat([t_feats, a_n, nu_n], dim=-1)

        out = self.temporal_net(inp).view(-1, 3, self.M)
        amp_raw = out[:, 0, :]
        xi_raw = out[:, 1, :]
        h_raw = out[:, 2, :]

        # Static anchors in [x_min, x_max]
        anchors = CFG.x_min + (CFG.x_max - CFG.x_min) * torch.sigmoid(self.anchor_raw).view(1, self.M)

        # Dynamic amplitudes
        alpha = torch.tanh(amp_raw) * self.amp_scale.view(1, self.M)

        # Dynamic centers in physical domain
        xi = anchors + 0.35 * torch.tanh(xi_raw)
        xi = torch.clamp(xi, CFG.x_min, CFG.x_max)

        # Dynamic widths in a safe bounded range
        h = self.h_min + (self.h_max - self.h_min) * torch.sigmoid(h_raw)

        return alpha, xi, h

    def basis_eval(self, x: torch.Tensor, t: torch.Tensor, a: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
        alpha, xi, h = self.dynamic_params(t, a, nu)
        x_exp = x.expand(-1, self.M)
        phi = torch.exp(-0.5 * ((x_exp - xi) / h) ** 2)
        v = torch.sum(alpha * phi, dim=1, keepdim=True)
        return v, (alpha, xi, h)

    def forward(self, xtanu: torch.Tensor, return_aux: bool = False):
        x = xtanu[:, 0:1]
        t = xtanu[:, 1:2]
        a = xtanu[:, 2:3]
        nu = xtanu[:, 3:4]

        u0 = initial_condition_torch(x, nu)
        v, aux = self.basis_eval(x, t, a, nu)
        u = u0 + t * v

        if return_aux:
            return u.squeeze(-1), aux
        return u.squeeze(-1)


# ============================================================
# Sampling utilities
# ============================================================
def sample_a_nu(N: int, nu_range: Tuple[float, float], device: torch.device):
    a = CFG.a_min + (CFG.a_max - CFG.a_min) * torch.rand(N, 1, device=device)
    # log-uniform for nu is better for resolving the lower-viscosity cases
    u = torch.rand(N, 1, device=device)
    log_min = math.log10(nu_range[0])
    log_max = math.log10(nu_range[1])
    nu = 10 ** (log_min + (log_max - log_min) * u)
    return a, nu


def sample_interior_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    x = CFG.x_min + (CFG.x_max - CFG.x_min) * torch.rand(N, 1, device=device)
    t = CFG.t_min + (CFG.t_max - CFG.t_min) * torch.rand(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)
    return torch.cat([x, t, a, nu], dim=1)


def sample_localized_interior_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    """
    Local points near the evolving Gaussian packet center.
    No characteristics are used in the model, but for sampling we may use the known
    benchmark center to focus collocation where the solution is active.
    This is similar in spirit to adaptive collocation around localized features.
    """
    t = CFG.t_min + (CFG.t_max - CFG.t_min) * torch.rand(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)

    center = CFG.x0_center + a * t
    width = 0.6 * torch.sqrt(nu * (4.0 * t + 1.0) + 1e-12)
    x = center + width * torch.randn_like(center)
    x = torch.clamp(x, CFG.x_min, CFG.x_max)
    return torch.cat([x, t, a, nu], dim=1)


def sample_ic_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    x = CFG.x_min + (CFG.x_max - CFG.x_min) * torch.rand(N, 1, device=device)
    t = torch.zeros(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)
    return torch.cat([x, t, a, nu], dim=1)


def sample_bc_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    t = CFG.t_min + (CFG.t_max - CFG.t_min) * torch.rand(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)
    xL = torch.full((N, 1), CFG.x_min, device=device)
    xR = torch.full((N, 1), CFG.x_max, device=device)
    left = torch.cat([xL, t, a, nu], dim=1)
    right = torch.cat([xR, t, a, nu], dim=1)
    return left, right


# ============================================================
# PDE residual
# ============================================================
def advecdiff_residual(model: nn.Module, xtanu: torch.Tensor) -> torch.Tensor:
    xtanu = xtanu.clone().detach().requires_grad_(True)
    u = model(xtanu)

    grads = torch.autograd.grad(
        outputs=u,
        inputs=xtanu,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]

    u_x = grads[:, 0]
    u_t = grads[:, 1]

    grads_x = torch.autograd.grad(
        outputs=u_x,
        inputs=xtanu,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
        retain_graph=True,
    )[0]
    u_xx = grads_x[:, 0]

    a = xtanu[:, 2]
    nu = xtanu[:, 3]

    return u_t + a * u_x - nu * u_xx


# ============================================================
# Evaluation / plotting
# ============================================================
def plot_training_history(history: Dict[str, List[float]], save_path: str):
    plt.figure(figsize=(7, 4.5))
    plt.plot(history["epoch"], history["loss"], lw=2, label="Total")
    plt.plot(history["epoch"], history["loss_pde"], lw=1.5, label="PDE")
    plt.plot(history["epoch"], history["loss_ic"], lw=1.5, label="IC")
    plt.plot(history["epoch"], history["loss_bc"], lw=1.5, label="BC")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training loss history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def evaluate_case_on_grid(model, a_test: float, nu_test: float, Nx: int = 200, Nt: int = 200, device=None):
    x = np.linspace(CFG.x_min, CFG.x_max, Nx)
    t = np.linspace(CFG.t_min, CFG.t_max, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")
    A = a_test * np.ones_like(X)
    NU = nu_test * np.ones_like(X)

    xtanu = np.stack([X.ravel(), T.ravel(), A.ravel(), NU.ravel()], axis=1)
    xtanu_t = torch.tensor(xtanu, dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        u_pred_flat = model(xtanu_t)

    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)
    u_exact = exact_advecdiff_np(X, T, a_test, nu_test)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    return {
        "x": x,
        "t": t,
        "X": X,
        "T": T,
        "u_exact": u_exact,
        "u_pred": u_pred,
        "abs_err": abs_err,
        "rel_l2": rel_l2,
    }


def plot_multicase_fields(model, test_cases, Nx: int, Nt: int, save_dir: str, device):
    num_tests = len(test_cases)
    fig, axs = plt.subplots(3, num_tests, figsize=(4.4 * num_tests, 9.6), constrained_layout=True)
    if num_tests == 1:
        axs = axs.reshape(3, 1)

    results = []
    for name, a_test, nu_test in test_cases:
        results.append((name, a_test, nu_test, evaluate_case_on_grid(model, a_test, nu_test, Nx=Nx, Nt=Nt, device=device)))

    all_sol_vals = np.concatenate([r[3]["u_exact"].ravel() for r in results] + [r[3]["u_pred"].ravel() for r in results])
    sol_vmin, sol_vmax = np.min(all_sol_vals), np.max(all_sol_vals)
    all_err_vals = np.concatenate([r[3]["abs_err"].ravel() for r in results])
    err_vmin, err_vmax = 0.0, np.max(all_err_vals)

    summary = []
    im_sol = None
    im_err = None
    for j, (name, a_test, nu_test, res) in enumerate(results):
        u_exact = res["u_exact"]
        u_pred = res["u_pred"]
        abs_err = res["abs_err"]
        rel_l2 = res["rel_l2"]
        summary.append({"name": name, "a": a_test, "nu": nu_test, "rel_l2": rel_l2})

        ax = axs[0, j]
        im_sol = ax.imshow(u_exact.T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_title(rf"$(a,\nu)=({a_test:.2f},{nu_test:.3f})$", fontsize=14)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Exact", fontsize=13)

        ax = axs[1, j]
        ax.imshow(u_pred.T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Prediction", fontsize=13)

        ax = axs[2, j]
        im_err = ax.imshow(abs_err.T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max], aspect="auto", vmin=err_vmin, vmax=err_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$|u_{pred}-u_{exact}|$", fontsize=13)

    cbar1 = fig.colorbar(im_sol, ax=axs[0:2, :], shrink=0.95, location="right")
    cbar1.set_label("Solution value", fontsize=12)
    cbar2 = fig.colorbar(im_err, ax=axs[2, :], shrink=0.95, location="right")
    cbar2.set_label("Absolute error", fontsize=12)

    training_text = rf"Training ranges: $a \in [{CFG.a_min:.2f},{CFG.a_max:.2f}]$, $\nu \in [{CFG.nu_min:.3f},{CFG.nu_max:.3f}]$"
    fig.text(0.5, -0.02, training_text, ha="center", va="top", fontsize=13)

    combined_path = os.path.join(save_dir, "multicase_fields.png")
    fig.savefig(combined_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return summary


def plot_time_slices(model, test_cases, t_slices, Nx: int, save_dir: str, device):
    x = np.linspace(CFG.x_min, CFG.x_max, Nx)
    for name, a_test, nu_test in test_cases:
        fig, axs = plt.subplots(1, len(t_slices), figsize=(4 * len(t_slices), 3.8), constrained_layout=True)
        if len(t_slices) == 1:
            axs = [axs]
        for k, t_val in enumerate(t_slices):
            xtanu = np.stack([x, t_val * np.ones_like(x), a_test * np.ones_like(x), nu_test * np.ones_like(x)], axis=1)
            xtanu_t = torch.tensor(xtanu, dtype=torch.float32, device=device)
            with torch.no_grad():
                u_pred = model(xtanu_t).cpu().numpy()
            u_exact = exact_advecdiff_np(x, t_val, a_test, nu_test)
            ax = axs[k]
            ax.plot(x, u_exact, lw=2, label="Exact")
            ax.plot(x, u_pred, "--", lw=2, label="Prediction")
            ax.set_title(f"t = {t_val:.2f}")
            ax.set_xlabel("x")
            ax.set_ylabel("u")
            ax.grid(True, alpha=0.3)
            if k == 0:
                ax.legend()
        fig.suptitle(f"Time slices: {name} | a={a_test:.3f}, nu={nu_test:.3f}", fontsize=13)
        save_path = os.path.join(save_dir, f"time_slices_{name}.png")
        fig.savefig(save_path, dpi=300)
        plt.close(fig)


def plot_error_summary(summary, save_dir: str):
    names = [row["name"] for row in summary]
    rel_l2 = [row["rel_l2"] for row in summary]
    plt.figure(figsize=(8, 4))
    plt.bar(names, rel_l2)
    plt.yscale("log")
    plt.ylabel("Relative L2 error")
    plt.title("Generalization across test cases")
    plt.xticks(rotation=25, ha="right")
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "error_summary.png"), dpi=300)
    plt.close()


def save_summary_csv(summary, save_dir: str):
    path = os.path.join(save_dir, "test_case_summary.csv")
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "a", "nu", "rel_l2"])
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


def visualize_dynamic_centers(model, test_cases, K: int, save_dir: str, device):
    model.eval()
    t_vals = np.linspace(CFG.t_min, CFG.t_max, K).astype(np.float32)
    for name, a_test, nu_test in test_cases:
        T = torch.tensor(t_vals[:, None], dtype=torch.float32, device=device)
        A = torch.full_like(T, a_test)
        NU = torch.full_like(T, nu_test)
        with torch.no_grad():
            alpha, xi, h = model.dynamic_params(T, A, NU)
        xi_np = xi.cpu().numpy()
        alpha_np = np.abs(alpha.cpu().numpy()) + 1e-12
        h_np = h.cpu().numpy()
        X_pts, T_pts, W_pts, S_pts = [], [], [], []
        for j in range(model.M):
            for k in range(K):
                X_pts.append(xi_np[k, j])
                T_pts.append(t_vals[k])
                W_pts.append(alpha_np[k, j])
                S_pts.append(250.0 * h_np[k, j])
        fig, ax = plt.subplots(figsize=(5.5, 5.2))
        sc = ax.scatter(X_pts, T_pts, c=W_pts, s=np.clip(S_pts, 8.0, 120.0), cmap="viridis", alpha=0.85)
        plt.colorbar(sc, ax=ax, label=r"$|\alpha_j(t,a,\nu)|$")
        ax.scatter([CFG.x0_center], [0.0], s=90, marker="x", color="red", label="IC center")
        ax.set_xlim(CFG.x_min, CFG.x_max)
        ax.set_ylim(CFG.t_min, CFG.t_max)
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(f"Dynamic basis centers\n{name}")
        ax.legend(loc="upper right")
        save_path = os.path.join(save_dir, f"dynamic_centers_{name}.png")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close(fig)


def run_full_evaluation(model, history, test_cases, Nx: int, Nt: int, save_dir: str, device):
    plot_training_history(history, os.path.join(save_dir, "training_loss.png"))
    summary = plot_multicase_fields(model, test_cases, Nx=Nx, Nt=Nt, save_dir=save_dir, device=device)
    plot_time_slices(model, test_cases, t_slices=(0.0, 0.125, 0.25, 0.375, 0.5), Nx=400, save_dir=save_dir, device=device)
    plot_error_summary(summary, save_dir)
    save_summary_csv(summary, save_dir)
    visualize_dynamic_centers(model, test_cases, K=60, save_dir=save_dir, device=device)
    return summary


# ============================================================
# Training
# ============================================================
def train_model(args, device):
    model = DynamicBasisMetaSPINN_AdvecDiff(M=args.M, hidden=args.hidden, num_freqs=args.num_freqs).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(args.epochs // 3, 1), gamma=0.5)

    best_loss = float("inf")
    best_state = None

    history = {
        "epoch": [],
        "loss": [],
        "loss_pde": [],
        "loss_ic": [],
        "loss_bc": [],
        "loss_reg": [],
    }

    for ep in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad()

        # viscosity curriculum
        if ep <= args.epochs // 2:
            nu_range = (CFG.nu_easy, CFG.nu_max)
        else:
            nu_range = (CFG.nu_min, CFG.nu_max)

        # interior: mixture of uniform + localized packet-aware samples
        xt_uniform = sample_interior_points(args.N_int_uniform, nu_range, device)
        xt_local = sample_localized_interior_points(args.N_int_local, nu_range, device)
        xt_int = torch.cat([xt_uniform, xt_local], dim=0)

        xt_ic = sample_ic_points(args.N_ic, nu_range, device)
        xtL, xtR = sample_bc_points(args.N_bc, nu_range, device)

        # PDE
        res = advecdiff_residual(model, xt_int)
        loss_pde = torch.mean(res ** 2)

        # IC (exactly hard-enforced by ansatz, but we still keep a tiny consistency term)
        u_ic = model(xt_ic)
        x_ic = xt_ic[:, 0:1]
        nu_ic = xt_ic[:, 3:4]
        u_ic_true = initial_condition_torch(x_ic, nu_ic).squeeze(-1)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        # BC from exact benchmark solution
        uL = model(xtL)
        uR = model(xtR)
        uL_true = exact_advecdiff_torch(xtL[:, 0:1], xtL[:, 1:2], xtL[:, 2:3], xtL[:, 3:4]).squeeze(-1)
        uR_true = exact_advecdiff_torch(xtR[:, 0:1], xtR[:, 1:2], xtR[:, 2:3], xtR[:, 3:4]).squeeze(-1)
        loss_bc = torch.mean((uL - uL_true) ** 2) + torch.mean((uR - uR_true) ** 2)

        # light regularization to avoid pathological widths / amplitudes
        aux_t = torch.rand(args.N_reg, 1, device=device) * (CFG.t_max - CFG.t_min) + CFG.t_min
        aux_a, aux_nu = sample_a_nu(args.N_reg, nu_range, device)
        alpha, _, h = model.dynamic_params(aux_t, aux_a, aux_nu)
        loss_reg = 1e-5 * torch.mean(alpha ** 2) + 1e-5 * torch.mean(1.0 / (h ** 2))

        loss = args.lambda_pde * loss_pde + args.lambda_ic * loss_ic + args.lambda_bc * loss_bc + loss_reg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        history["epoch"].append(ep)
        history["loss"].append(loss.item())
        history["loss_pde"].append(loss_pde.item())
        history["loss_ic"].append(loss_ic.item())
        history["loss_bc"].append(loss_bc.item())
        history["loss_reg"].append(loss_reg.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % args.print_every == 0 or ep == 1:
            print(
                f"Epoch {ep:5d} | Loss {loss.item():.3e} | "
                f"PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | "
                f"BC {loss_bc.item():.3e} | Reg {loss_reg.item():.3e} | "
                f"nu_range=[{nu_range[0]:.3f},{nu_range[1]:.3f}]"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser(description="Dynamic-basis meta-SPINN for parametric advection-diffusion")
    parser.add_argument("--gpu", action="store_true", help="Use CUDA if available")
    parser.add_argument("--epochs", type=int, default=4000)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--M", type=int, default=48)
    parser.add_argument("--hidden", type=int, default=128)
    parser.add_argument("--num_freqs", type=int, default=4)
    parser.add_argument("--N_int_uniform", type=int, default=1024)
    parser.add_argument("--N_int_local", type=int, default=1024)
    parser.add_argument("--N_ic", type=int, default=256)
    parser.add_argument("--N_bc", type=int, default=256)
    parser.add_argument("--N_reg", type=int, default=128)
    parser.add_argument("--lambda_pde", type=float, default=1.0)
    parser.add_argument("--lambda_ic", type=float, default=1.0)
    parser.add_argument("--lambda_bc", type=float, default=1.0)
    parser.add_argument("--Nx", type=int, default=200)
    parser.add_argument("--Nt", type=int, default=200)
    parser.add_argument("--print_every", type=int, default=100)
    parser.add_argument("--outdir", type=str, default="figures_meta_advecdiff_dynamic")
    parser.add_argument("--model_path", type=str, default="meta_spinn_advecdiff_dynamic.pt")
    args, _unknown = parser.parse_known_args()

    device = torch.device("cuda" if (args.gpu and torch.cuda.is_available()) else "cpu")
    print("Using device:", device)
    os.makedirs(args.outdir, exist_ok=True)

    model, history = train_model(args, device)
    torch.save(model.state_dict(), os.path.join(args.outdir, args.model_path))
    print(f"Model saved to: {os.path.join(args.outdir, args.model_path)}")

    loaded_model = DynamicBasisMetaSPINN_AdvecDiff(M=args.M, hidden=args.hidden, num_freqs=args.num_freqs).to(device)
    loaded_model.load_state_dict(torch.load(os.path.join(args.outdir, args.model_path), map_location=device))
    loaded_model.eval()

    test_cases = [
        ("mid_range", 0.75, 0.030),
        ("low_a_high_nu", 0.55, 0.045),
        ("high_a_low_nu", 0.95, 0.015),
        ("nu_out_of_range_low", 0.75, 0.008),
    ]

    summary = run_full_evaluation(loaded_model, history, test_cases, Nx=args.Nx, Nt=args.Nt, save_dir=args.outdir, device=device)

    print("\nSummary of test cases:")
    for row in summary:
        print(f"{row['name']:22s} | a={row['a']:.3f}, nu={row['nu']:.3f} | relL2={row['rel_l2']:.3e}")
    print(f"\nAll outputs saved in: {args.outdir}")


if __name__ == "__main__":
    main()

Using device: cpu
Epoch     1 | Loss 5.143e+00 | PDE 5.082e+00 | IC 0.000e+00 | BC 5.988e-02 | Reg 2.989e-04 | nu_range=[0.030,0.050]
Epoch   100 | Loss 1.815e-01 | PDE 1.769e-01 | IC 0.000e+00 | BC 3.791e-03 | Reg 8.756e-04 | nu_range=[0.030,0.050]
Epoch   200 | Loss 7.232e-02 | PDE 7.000e-02 | IC 0.000e+00 | BC 1.433e-03 | Reg 8.890e-04 | nu_range=[0.030,0.050]
Epoch   300 | Loss 5.643e-02 | PDE 5.472e-02 | IC 0.000e+00 | BC 8.625e-04 | Reg 8.484e-04 | nu_range=[0.030,0.050]
Epoch   400 | Loss 4.037e-02 | PDE 3.883e-02 | IC 0.000e+00 | BC 7.031e-04 | Reg 8.393e-04 | nu_range=[0.030,0.050]
Epoch   500 | Loss 1.898e-02 | PDE 1.770e-02 | IC 0.000e+00 | BC 4.805e-04 | Reg 8.005e-04 | nu_range=[0.030,0.050]
Epoch   600 | Loss 2.153e-02 | PDE 2.044e-02 | IC 0.000e+00 | BC 2.874e-04 | Reg 8.015e-04 | nu_range=[0.030,0.050]
Epoch   700 | Loss 1.573e-02 | PDE 1.475e-02 | IC 0.000e+00 | BC 2.367e-04 | Reg 7.411e-04 | nu_range=[0.030,0.050]
Epoch   800 | Loss 1.409e-02 | PDE 1.320e-02 | IC 0.00

/tmp/ipykernel_71052/2178833780.py:613: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load(os.path.join(args.outdir, args.model_path), map


Summary of test cases:
mid_range              | a=0.750, nu=0.030 | relL2=7.644e-03
low_a_high_nu          | a=0.550, nu=0.045 | relL2=1.055e-02
high_a_low_nu          | a=0.950, nu=0.015 | relL2=1.924e-01
nu_out_of_range_low    | a=0.750, nu=0.008 | relL2=3.904e-01

All outputs saved in: figures_meta_advecdiff_dynamic
